# Object Detection

Use votenet from openMMlab to detect objects in 3D scans

In [ ]:
import torch
print("torch is installed:")
if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))
    print("CUDA device count:", torch.cuda.device_count())
else:
    print("No CUDA GPU detected")
print("torch version: " + torch.__version__)
print("torch-cuda version: " + torch.version.cuda)

## Folder Votenet Detection

In [ ]:
import os
demoFile = r"/home/jvermandere/projects/mmdetection3d/demo/pcd_folder_demo.py"
folder = r"/home/jvermandere/datasets/V-Scan/data"
configFile = r"/home/jvermandere/projects/mmdetection3d/configs/votenet/votenet_8xb8_scannet-3d.py"
weightsFile = r"/home/jvermandere/projects/DRM/_weights/votenet_8x8_scannet-3d-18class_20210823_234503-cf8134fa.pth"
scoreThr = 0.1
filename = "main.bin"

#configFile = r"/home/jvermandere/projects/mmdetection3d/configs/votenet/votenet_vscan.py"
#weightsFile = r"/home/jvermandere/projects/DRM/_weights/votenet_vscan_best.pth"


In [ ]:

command = f'python {demoFile} {folder} {configFile} {weightsFile} --pred-score-thr {scoreThr} --filename {filename}'
print("running command: " + command)

os.system(command)

## Object detection evaluation

In [ ]:
import numpy as np
import trimesh
from scipy.optimize import linear_sum_assignment


def mesh_iou(mesh_a, mesh_b):
    inter = trimesh.boolean.intersection([mesh_a, mesh_b])
    if inter is None or inter.volume == 0:
        return 0.0

    union = trimesh.boolean.union([mesh_a, mesh_b])
    return inter.volume / union.volume


def compute_iou_matrix(pred_meshes, gt_meshes):
    iou_matrix = np.zeros((len(pred_meshes), len(gt_meshes)))

    for i, p in enumerate(pred_meshes):
        for j, g in enumerate(gt_meshes):
            iou_matrix[i, j] = mesh_iou(p, g)

    return iou_matrix


def evaluate_meshes(pred_meshes, gt_meshes, iou_threshold=0.5):

    iou_matrix = compute_iou_matrix(pred_meshes, gt_meshes)

    # Hungarian matching (maximize IoU)
    pred_idx, gt_idx = linear_sum_assignment(-iou_matrix)

    matches = []
    tp = 0
    matched_ious = []

    for p, g in zip(pred_idx, gt_idx):
        iou = iou_matrix[p, g]
        matches.append((p, g, iou))

        if iou >= iou_threshold:
            tp += 1
            matched_ious.append(iou)

    fp = len(pred_meshes) - tp
    fn = len(gt_meshes) - tp

    precision = tp / (tp + fp) if (tp + fp) else 0
    recall = tp / (tp + fn) if (tp + fn) else 0

    avg_iou = np.mean(matched_ious) if matched_ious else 0

    return {
        "precision": precision,
        "recall": recall,
        "average_iou": avg_iou,
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "matches": matches
    }


In [ ]:
from pathlib import Path
from collections import defaultdict
import sys
sys.path.insert(0, '../')
import drm
from drm import detect

root = Path(folder)
matches = list(root.rglob(filename))
pcd_files = sorted(matches)

print(f'Found {len(pcd_files)} file(s) named "{filename}"')

all_metrics = []
class_metrics = defaultdict(list)

def get_scan_class(pcd_path):
    """Extract class by stripping the trailing _<numbers> from the folder name."""
    folder_name = pcd_path.parent.name
    parts = folder_name.rsplit('_', 1)
    # Only strip the suffix if it's purely numeric
    if len(parts) == 2 and parts[1].isdigit():
        return parts[0]
    return folder_name

for i, pcd_path in enumerate(pcd_files):
    print(f'[{i + 1}/{len(pcd_files)}] Comparing {pcd_path}')
    bb_detected = (pcd_path.parent / "preds" / pcd_path.stem).with_suffix(".json")
    bb_gt = pcd_path.as_posix()[:-4] + "_bb.json"
    gt_meshes = drm.detect.load_gt_json_boxes_as_mesh(bb_gt)
    detected_meshes = drm.detect.load_detected_boxes_as_mesh(bb_detected, pred_score_thr=scoreThr)
    metrics = evaluate_meshes(detected_meshes, gt_meshes, 0.3)
    metrics["id"] = pcd_path.parent.name

    scan_class = get_scan_class(pcd_path)
    all_metrics.append(metrics)
    class_metrics[scan_class].append(metrics)

    for p, g, iou in metrics["matches"]:
        pass

# Per-class averages
print("\n=== Average metrics per class ===")
for scan_class, metrics_list in sorted(class_metrics.items()):
    avg_precision = sum(m["precision"] for m in metrics_list) / len(metrics_list)
    avg_recall = sum(m["recall"] for m in metrics_list) / len(metrics_list)
    avg_iou = sum(m["average_iou"] for m in metrics_list) / len(metrics_list)
    print(f"\n  [{scan_class}] ({len(metrics_list)} scan(s))")
    print(f"  Avg Precision : {avg_precision:.4f}")
    print(f"  Avg Recall    : {avg_recall:.4f}")
    print(f"  Avg IoU       : {avg_iou:.4f}")

# Overall averages
avg_precision = sum(m["precision"] for m in all_metrics) / len(all_metrics)
avg_recall = sum(m["recall"] for m in all_metrics) / len(all_metrics)
avg_iou = sum(m["average_iou"] for m in all_metrics) / len(all_metrics)

print("\n=== Average metrics over all scans ===")
print(f"Scans evaluated : {len(all_metrics)}")
print(f"Avg Precision   : {avg_precision:.4f}")
print(f"Avg Recall      : {avg_recall:.4f}")
print(f"Avg IoU         : {avg_iou:.4f}")

In [ ]:
# ---------------------------------------------------------------------------
# Save metrics to JSON
# ---------------------------------------------------------------------------
import json
from datetime import datetime

def make_serializable(obj):
    """Recursively convert numpy types and other non-serializable objects."""
    import numpy as np
    if isinstance(obj, dict):
        return {k: make_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, (list, tuple)):
        return [make_serializable(v) for v in obj]
    elif isinstance(obj, np.integer):
        return int(obj)
    elif isinstance(obj, np.floating):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj

# Build output structure
output = {
    "timestamp":   datetime.now().isoformat(),
    "config": {
        "score_threshold": scoreThr,
        "iou_threshold":   0.3,
        "filename":        filename,
        "folder":          str(folder),
    },
    "overall": {
        "num_scans":     len(all_metrics),
        "avg_precision": avg_precision,
        "avg_recall":    avg_recall,
        "avg_iou":       avg_iou,
    },
    "per_class": {
        scan_class: {
            "num_scans":     len(metrics_list),
            "avg_precision": sum(m["precision"]    for m in metrics_list) / len(metrics_list),
            "avg_recall":    sum(m["recall"]        for m in metrics_list) / len(metrics_list),
            "avg_iou":       sum(m["average_iou"]   for m in metrics_list) / len(metrics_list),
        }
        for scan_class, metrics_list in sorted(class_metrics.items())
    },
    "per_scan": make_serializable(all_metrics),
}

# Save
timestamp_str = datetime.now().strftime("%Y%m%d_%H%M%S")
out_path = Path("../_output/object_detection_results") / f"metrics_{timestamp_str}.json"
with open(out_path, "w") as f:
    json.dump(output, f, indent=2)

print(f"\n✓ Metrics saved → {out_path}")

### Visualisation

In [ ]:
scoreThr = 0.15
pcd_path = Path(r"/home/jvermandere/datasets/V-Scan/data/Office_1_Leica-P30_1775812781853/main.bin")
bb_detected = (pcd_path.parent / "preds" / pcd_path.stem).with_suffix(".json")
bb_gt = pcd_path.as_posix()[:-4] + "_bb.json"
gt_meshes = drm.detect.load_gt_json_boxes_as_mesh(bb_gt, randomcolors=False)
detected_meshes = drm.detect.load_detected_boxes_as_mesh(bb_detected, pred_score_thr=scoreThr, labelColors=True)
pcd = drm.load_bin_pointcloud(str(pcd_path))
points = np.asarray(pcd.vertices)
indices = np.random.choice(len(points), size=10000, replace=False)
sampled = points[indices]
pcd = trimesh.PointCloud(sampled)
trimesh.Scene([detected_meshes, pcd]).show()

In [ ]:
from pathlib import Path

root = Path(folder)
matches = list(root.rglob(filename))
pcd_files = sorted(matches)

print(f'Found {len(pcd_files)} file(s) named "{filename}"')

for i, pcd_path in enumerate(pcd_files):
    # Save results in the same subfolder as the point cloud file
    out_dir = str(pcd_path.parent)
    print(f'[{i + 1}/{len(pcd_files)}] Running detection on {pcd_path}')

In [ ]:
from pathlib import Path

binPath = Path("/home/jvermandere/projects/DRM/_input/virtualDataset/VirtualScanner-1773153754053/main.bin")#"/home/jvermandere/projects/DRM/_input/Office2.bin"
# Define the output directory
bb_detected = (binPath.parent / "results" / "preds" / binPath.stem).with_suffix(".json")
bb_gt = binPath.as_posix()[:-4] + "_bb.txt"

## Run as package

In [ ]:
import sys
sys.path.insert(0, '../')
import drm

import drm.detect

In [ ]:
# Load the model once
inferencer = drm.detect.create_inferencer(
    model_config='/home/jvermandere/projects/mmdetection3d/configs/votenet/votenet_8xb8_scannet-3d.py',
    weights='/home/jvermandere/projects/DRM/_weights/votenet_8x8_scannet-3d-18class_20210823_234503-cf8134fa.pth',
)


In [ ]:

# Run detection on as many point clouds as you need
results_1 = drm.detect.run_detection(inferencer, '/home/jvermandere/projects/DRM/_input/virtualDataset/VirtualScanner-1773153754053/main.bin')

In [ ]:
import numpy as np
import torch
from mmengine.dataset import pseudo_collate
from mmdet3d.apis import init_model
from mmdet3d.structures import LiDARInstance3DBoxes

CONFIG = "../_weights_and_configs/configs/votenet/votenet_8xb8_scannet-3d.py"
CHECKPOINT = "../_weights_and_configs/votenet_8x8_scannet-3d-18class_20210823_234503-cf8134fa.pth"

# Load your ASCII or .bin point cloud
points = np.fromfile('../_input/000008.bin', dtype=np.float32).reshape(-1,4)
xyz = points[:, :3]
rgb = np.zeros_like(xyz)         # dummy RGB if none
pc_input = np.concatenate([xyz, rgb], axis=1).astype(np.float32)

# Wrap in a dict that matches the ScanNet dataset format
data = dict(
    points=torch.from_numpy(pc_input),
    pts_filename='',                    # required key
    box_type_3d=LiDARInstance3DBoxes   # for VoteNet
)

# Collate into a batch
batch = pseudo_collate([data])

# Init model
model = init_model(CONFIG, CHECKPOINT, device='cuda:0')

# Inference
with torch.no_grad():
    result = model.test_step(batch)

# Extract results
boxes_3d = result[0]['boxes_3d']
scores_3d = result[0]['scores_3d']
labels_3d = result[0]['labels_3d']

print("Detected boxes:", len(boxes_3d))

In [ ]:
PcdPath = "/home/jelle-vermandere/Documents/Github/DRM/_input/Office1 - Cloud.txt"

In [ ]:
import numpy as np
import tempfile
from mmdet3d.apis import init_model, inference_detector

# -----------------------------
# SETTINGS
# -----------------------------
CONFIG = "../_weights_and_configs/configs/votenet/votenet_8xb8_scannet-3d.py"
CHECKPOINT = "../_weights_and_configs/votenet_8x8_scannet-3d-18class_20210823_234503-cf8134fa.pth"
ASCII_FILE = "../_input/Office1 - Cloud.txt"


In [ ]:

# -----------------------------
# LOAD ASCII POINT CLOUD
# -----------------------------
points = np.loadtxt(ASCII_FILE)

xyz = points[:, 0:3]
rgb = points[:, 6:9] / 255.0   # normalize colors (important)

# VoteNet expects XYZRGB
pc = np.concatenate([xyz, rgb], axis=1)

# -----------------------------
# LOAD MODEL
# -----------------------------
model = init_model(CONFIG, CHECKPOINT, device='cuda:0')


In [ ]:
print(pc[:100].shape)

In [ ]:
import numpy as np
import torch
from mmengine.dataset import pseudo_collate
from mmdet3d.apis import init_model
# prepare input
points = pc[:100].astype(np.float32)
pointsDict = dict(points=torch.from_numpy(points))
# collate (simulate batch dimension)
collated = pseudo_collate([pointsDict])
# inference
with torch.no_grad():
    result = model.test_step(collated)
# -----------------------------
# RUN INFERENCE
# -----------------------------
#result, data = inference_detector(model, collated)

print(result)

In [ ]:
import open3d as o3d
import numpy as np
import json

# Example data
with open("/home/jelle-vermandere/Documents/Github/DRM/_output/scene0000_00.json") as f:
    data = json.load(f)
    print(data)


# Parameters
score_threshold = 0.8  # Only show boxes with score > threshold

# Colormap for labels
label_colors = [
    [1, 0, 0],  # red
    [0, 1, 0],  # green
    [0, 0, 1],  # blue
    [1, 1, 0],  # yellow
    [1, 0, 1],  # magenta
    [0, 1, 1],  # cyan
]

def create_bbox(center, size, rotation_z=0.0, color=[1,0,0]):
    """
    Create an Open3D OrientedBoundingBox from center, size, rotation, and color
    """
    bbox = o3d.geometry.OrientedBoundingBox()
    bbox.center = center
    bbox.extent = size
    R = o3d.geometry.get_rotation_matrix_from_axis_angle([0, 0, rotation_z])
    bbox.R = R
    bbox.color = color
    return bbox

# Create Open3D geometries
geometries = []

for label, score, box in zip(data["labels_3d"], data["scores_3d"], data["bboxes_3d"]):
    if score < score_threshold:
        continue

    # box = [x, y, z, dx, dy, dz, rotation_z]
    center = np.array(box[:3])
    size = np.array(box[3:6])
    rotation_z = box[6]
    color = label_colors[label % len(label_colors)]
    bbox = create_bbox(center, size, rotation_z, color)
    geometries.append(bbox)

# Add coordinate frame
geometries.append(o3d.geometry.TriangleMesh.create_coordinate_frame(size=1.0))


In [ ]:

# Visualize
o3d.visualization.draw_geometries(geometries)

In [ ]:
import trimesh
import numpy as np

# Example data
# Example data
with open("/home/jelle-vermandere/Documents/Github/DRM/_output/000017.json") as f:
    data = json.load(f)
    print(data)

# Parameters
score_threshold = 0.5  # Only show boxes with score > threshold

# Colormap for labels
label_colors = [
    [1, 0, 0, 0.5],  # red, alpha 0.5
    [0, 1, 0, 0.5],  # green
    [0, 0, 1, 0.5],  # blue
    [1, 1, 0, 0.5],  # yellow
    [1, 0, 1, 0.5],  # magenta
    [0, 1, 1, 0.5],  # cyan
]

def create_trimesh_box(center, size, rotation_z=0.0, color=[1,0,0,0.5]):
    """
    Create a trimesh Box mesh with given center, size, rotation, and color.
    """
    # Box is created centered at origin
    box = trimesh.creation.box(extents=size, transform=None)
    
    # Rotation matrix around z-axis
    c, s = np.cos(rotation_z), np.sin(rotation_z)
    R = np.array([
        [c, -s, 0, 0],
        [s,  c, 0, 0],
        [0,  0, 1, 0],
        [0,  0, 0, 1]
    ])
    
    # Translation to center
    T = np.eye(4)
    T[:3, 3] = center

    # Apply transform
    box.apply_transform(T @ R)
    
    # Set color (RGBA)
    box.visual.face_colors = color
    
    return box

# Create list of meshes
meshes = []

for label, score, box in zip(data["labels_3d"], data["scores_3d"], data["bboxes_3d"]):
    if score < score_threshold:
        continue

    center = np.array(box[:3])
    size = np.array(box[3:6])
    rotation_z = box[6]
    color = label_colors[label % len(label_colors)]
    
    mesh = create_trimesh_box(center, size, rotation_z, color)
    meshes.append(mesh)



In [ ]:
def load_bin_pointcloud(file_path):
    """Load KITTI-style .bin point cloud"""
    points = np.fromfile(file_path, dtype=np.float32).reshape(-1, 6)  # x, y, z, rgb
    # Normalize reflectance to [0,255] for colors
    reflectance = points[:, 3]
    colors = np.stack([reflectance, reflectance, reflectance], axis=1)  # grayscale
    colors = (colors / colors.max() * 255).astype(np.uint8)
    cloud = trimesh.points.PointCloud(points[:, :3], colors=points[:, 3:6]/255)
    return cloud

# -----------------------------
# Load point cloud
# -----------------------------
pointcloud_file = "/home/jelle-vermandere/Documents/Github/DRM/_input/000017.bin"  # Replace with your path
pcd = load_bin_pointcloud(pointcloud_file)
meshes.append(pcd)

In [ ]:
# Combine meshes for visualization
scene = trimesh.Scene(meshes)

# Show interactive visualization
scene.show()

## TR3D

In [ ]:
import os
demoFile = r"/home/jvermandere/projects/mmdetection3d/demo/pcd_folder_demo.py"
folder = r"/home/jvermandere/datasets/V-Scan/data"
configFile = r"/home/jvermandere/projects/mmdetection3d/configs/votenet/votenet_8xb8_scannet-3d.py"
weightsFile = r"/home/jvermandere/projects/DRM/_weights/votenet_8x8_scannet-3d-18class_20210823_234503-cf8134fa.pth"
scoreThr = 0.1
filename = "main.bin"

configFile = r"/home/jvermandere/projects/mmdetection3d/projects/TR3D/configs/tr3d_1xb16_scannet-3d-18class.py"
weightsFile = r"/home/jvermandere/projects/DRM/_weights/tr3d_1xb16_scannet-3d-18class.pth"

command = f'python {demoFile} {folder} {configFile} {weightsFile} --pred-score-thr {scoreThr} --filename {filename}'
print("running command: " + command)

os.system(command)


In [ ]:
!pip install git+https://github.com/NVIDIA/MinkowskiEngine -v --no-deps --install-option="--blas_include_dirs=${CONDA_PREFIX}/include" --install-option="--blas=openblas"
